# Local PDF Download + PyMuPDF Extraction

Runs steps 1 and 2 of the pipeline on your **laptop** (no GPU needed). PDFs land
in the same Google Drive folder Colab uses, so the existing
`colab_gpu_extraction.ipynb` can pick up at step 3 (Qwen2-VL on scanned PDFs)
without changes.

## Why this notebook exists
Cloudflare blocks SF court requests from Colab IPs but accepts them from your
laptop (where you solved the verification challenge in your browser). Running
the download step locally sidesteps the 403s. The GPU step still needs Colab.

## Prereqs
1. **Google Drive for Desktop** installed and signed in — confirm `~/Library/CloudStorage/GoogleDrive-<email>/My Drive/litigation-pipeline/` exists.
2. `conda activate ML` (or any env with this repo `pip install -e`'d).
3. A fresh SessionID from your browser when prompted (see Step 1 instructions).

## Handoff to Colab
After this notebook finishes, switch to `colab_gpu_extraction.ipynb` and run
from **cell-9 (Step 3 — Load Vision Model)** onward. Cells 1–7 in that notebook
are already done by this one.

## Setup

Edit `DRIVE_PATH` below if your Google account email is different from the
default. The cell will hard-error if the path doesn't exist so you don't
accidentally write to the wrong location.

In [ ]:
import sys
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================
# Drive Desktop sync path. Adjust the email if yours differs.
DRIVE_PATH = Path(
    "/Users/ernesto/Library/CloudStorage/GoogleDrive-ernestod1998@gmail.com"
    "/My Drive/litigation-pipeline"
)

# Local repo path — needed so we can import scraper modules.
REPO_DIR = Path("/Users/ernesto/Desktop/MLOPS-Project/litigation-outcome-pipeline")
# ============================================================

if not DRIVE_PATH.exists():
    raise RuntimeError(
        f"Drive Desktop sync not found at:\n  {DRIVE_PATH}\n\n"
        "Install Google Drive for Desktop from https://www.google.com/drive/download/, "
        "sign in with the same Google account you use in Colab, and confirm the path appears."
    )

if not REPO_DIR.exists():
    raise RuntimeError(f"Repo not found at {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

PDF_DIR = DRIVE_PATH / "pdfs"
OUTPUT_DIR = DRIVE_PATH / "extracted"
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

existing_pdfs = list(PDF_DIR.glob("*.pdf"))
existing_cases = {p.name.split("_")[0] for p in existing_pdfs if p.name.startswith("CSM")}

print(f"Drive PDF dir:    {PDF_DIR}")
print(f"Drive output dir: {OUTPUT_DIR}")
print(f"Existing PDFs:    {len(existing_pdfs)}")
print(f"Cases already covered: {len(existing_cases)} (these will be skipped on download)")

## Step 1 — Download PDFs from the court API

Reads `valid_cases.json` from the local repo and downloads documents for every
case not already in Drive. Only whitelisted document types are downloaded
(claims, judgments, orders, dismissals); procedural docs are skipped.

**To get a SessionID:**
1. Open https://webapps.sftc.org/cc/CaseCalendar.dll in your browser
2. Solve the Cloudflare verification (one click)
3. Copy the hex string after `SessionID=` in the URL

Both result-code `-1` (API session expired) and HTTP 403 (Cloudflare reject)
trigger a re-prompt for a fresh SessionID.

In [ ]:
import json
import subprocess
import time

import requests

from scraper.config import is_doc_type_wanted as is_doc_wanted
from scraper.court_api import sanitize_description as sanitize

BASE_URL = "https://webapps.sftc.org"
CASE_PATH = "/ci/CaseInfo.dll"
USER_AGENT = "MSDS603-Research-Scraper/1.0 (SF Small Claims academic study)"
DOWNLOAD_DELAY = 2.5  # seconds between requests


def alert(msg="Session ID expired. Paste new one."):
    """Audible + macOS notification banner so the user can step away from the keyboard."""
    try:
        subprocess.Popen(["say", msg])
    except Exception:
        pass
    try:
        subprocess.Popen(
            [
                "osascript",
                "-e",
                f'display notification "{msg}" with title "Court Scraper" sound name "Glass"',
            ]
        )
    except Exception:
        pass


def ask_session_id():
    alert()
    print("\n" + "=" * 50)
    print("SESSION ID NEEDED")
    print("1. Open https://webapps.sftc.org/cc/CaseCalendar.dll")
    print("2. Copy the hex string after SessionID= from the URL")
    print("=" * 50)
    return input("Paste SessionID: ").strip()


def is_session_error(exc):
    """Treat both the API's -1 reply and Cloudflare 403s as session expiry."""
    msg = str(exc)
    if "SESSION_EXPIRED" in msg:
        return True
    if "403" in msg and "Forbidden" in msg:
        return True
    return False


def get_documents(case_num, session_id):
    url = (
        f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetDocuments/"
        f"{case_num}/{session_id}/"
    )
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if data["result"][0] == -1:
        raise Exception("SESSION_EXPIRED")
    if data["result"][0] == 0:
        return []
    return json.loads(data["result"][1])


# Load valid case numbers from the repo
valid_cases_path = REPO_DIR / "scraper" / "state" / "valid_cases.json"
with open(valid_cases_path) as f:
    store = json.load(f)
all_valid = sorted(store.get("valid", {}).keys())
print(f"Valid cases from repo: {len(all_valid)}", flush=True)

# Build skip set ONCE — cheaper than 2664 separate globs in Drive Desktop stream mode
print("Listing existing PDFs in Drive (one-time)...", flush=True)
t0 = time.time()
existing_pdf_files = list(PDF_DIR.glob("*.pdf"))
existing_cases = {p.name.split("_")[0] for p in existing_pdf_files if p.name.startswith("CSM")}
print(
    f"  {len(existing_pdf_files)} PDFs across {len(existing_cases)} cases "
    f"(took {time.time() - t0:.1f}s)",
    flush=True,
)

to_download = [c for c in all_valid if c not in existing_cases]
print(f"Already in Drive (skipped): {len(all_valid) - len(to_download)}", flush=True)
print(f"To download:                {len(to_download)}", flush=True)

if to_download:
    session_id = ask_session_id()
    http = requests.Session()
    http.headers["User-Agent"] = USER_AGENT
    total_pdfs = 0

    for i, case_num in enumerate(to_download, start=1):
        if case_num in existing_cases:
            continue

        # Retry loop so a session refresh doesn't skip the case
        docs = None
        while True:
            try:
                time.sleep(DOWNLOAD_DELAY)
                print(f"[{i}/{len(to_download)}] {case_num}: fetching doc list...", flush=True)
                docs = get_documents(case_num, session_id)
                break
            except Exception as e:
                if is_session_error(e):
                    print(f"  Session rejected ({type(e).__name__}); refreshing", flush=True)
                    session_id = ask_session_id()
                    continue
                print(f"  Error: {e}", flush=True)
                docs = None
                break

        if docs is None:
            continue

        wanted = [
            d for d in docs
            if d.get("URL") and is_doc_wanted(d.get("DESCRIPTION", "doc"))
        ]
        print(f"  -> {len(docs)} total docs, {len(wanted)} wanted", flush=True)

        case_pdfs = 0
        for doc in wanted:
            desc = doc["DESCRIPTION"]
            pdf_path = PDF_DIR / f"{case_num}_{sanitize(desc)}.pdf"
            if pdf_path.exists():
                print(f"  - {desc[:50]}: already exists", flush=True)
                continue

            time.sleep(DOWNLOAD_DELAY)
            try:
                resp = http.get(doc["URL"], timeout=60, stream=True)
                if "pdf" in resp.headers.get("content-type", "").lower():
                    with open(pdf_path, "wb") as out:
                        for chunk in resp.iter_content(8192):
                            out.write(chunk)
                    size_kb = pdf_path.stat().st_size / 1024
                    print(f"  + {desc[:50]}: {size_kb:.0f} KB", flush=True)
                    total_pdfs += 1
                    case_pdfs += 1
                else:
                    ct = resp.headers.get("content-type", "?")
                    print(f"  ! {desc[:50]}: not a PDF (content-type: {ct})", flush=True)
            except Exception as e:
                print(f"  ! {desc[:50]}: download failed: {e}", flush=True)

        existing_cases.add(case_num)
        print(
            f"  = {case_num} done: {case_pdfs}/{len(wanted)} new PDFs "
            f"(total this run: {total_pdfs})",
            flush=True,
        )

    print(f"\nDownloaded {total_pdfs} PDFs total")
else:
    print("All PDFs already in Drive.")

all_pdfs = sorted(p.name for p in PDF_DIR.glob("*.pdf"))
print(f"\nTotal PDFs in Drive: {len(all_pdfs)}")

## Step 2 — PyMuPDF (free text extraction)

Extracts the text layer from each PDF instantly. No GPU needed. PDFs that
are scanned images (<100 chars of text) get queued for the GPU step on Colab.
Corrupt PDFs (partial downloads / HTML error pages) are skipped with a log line.

In [ ]:
import fitz  # PyMuPDF

MIN_TEXT_LENGTH = 100


def extract_with_pymupdf(pdf_path):
    doc = fitz.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages).strip()


all_pdfs = sorted(PDF_DIR.glob("*.pdf"))

# Only extract whitelisted document types
pdfs = [
    p
    for p in all_pdfs
    if is_doc_wanted(p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem)
]
skipped_types = len(all_pdfs) - len(pdfs)

already_done = []
pymupdf_extracted = []
needs_gpu = []
corrupt = []

for pdf_path in pdfs:
    txt_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"

    if txt_path.exists():
        already_done.append(pdf_path)
        continue

    try:
        text = extract_with_pymupdf(pdf_path)
    except Exception as e:
        size = pdf_path.stat().st_size if pdf_path.exists() else 0
        print(f"  CORRUPT ({size} bytes): {pdf_path.name} — {type(e).__name__}: {e}")
        corrupt.append(pdf_path)
        continue

    if len(text) > MIN_TEXT_LENGTH:
        txt_path.write_text(text, encoding="utf-8")
        pymupdf_extracted.append(pdf_path)
    else:
        needs_gpu.append(pdf_path)

print(f"Skipped (not in whitelist):  {skipped_types}")
print(f"Already extracted (skipped): {len(already_done)}")
print(f"PyMuPDF extracted this run:  {len(pymupdf_extracted)}")
print(f"Need GPU vision model:       {len(needs_gpu)}")
print(f"Corrupt (skipped):           {len(corrupt)}")
if corrupt:
    print("\nTo re-download corrupt PDFs:")
    print("  for p in corrupt: p.unlink()")
    print("Then re-run Step 1.")
if needs_gpu:
    print(f"\nNext: open colab_gpu_extraction.ipynb and run from cell-9 onward to handle the {len(needs_gpu)} scanned PDFs.")